# 07 Realistic Track A End-To-End Validation

This notebook is the realistic end-to-end Track A layer for the downstream Dutch 15-minute DAM extension.

Current scope:

- bridge the hourly anchor target series into the post-implementation quarter-hour regime;
- check whether the selected hourly FS3 anchor models can actually be extended repo-natively into the observed Jan-Apr 2026 window;
- if feasible, run the realistic hourly-anchor extension and re-evaluate the 15-minute shape layer end to end;
- if not feasible, surface the blocking external-feature coverage gap explicitly instead of silently falling back to an oracle anchor.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path
    for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
    if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)

PACKAGE_ROOT = REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from quarterhour_da import (
    QuarterHourDAExtensionConfig,
    assert_thesis_grade_actual_source_authorized,
    build_thesis_grade_frozen_actual_metadata,
    find_frozen_actual_version,
    find_latest_canonical_actual_run,
    find_latest_observed_deterministic_run,
    find_latest_phase01_run,
    find_latest_phase02_run,
    find_latest_phase03_run,
    find_latest_phase04_run,
    find_latest_phase07_run,
    find_latest_phase07_upstream_refresh_run,
    load_frozen_actual_diagnostics,
    load_frozen_actual_manifest,
    load_frozen_actual_path,
    resolve_frozen_actual_registry_entry,
    run_observed_market_deterministic_forecast,
)

config = QuarterHourDAExtensionConfig()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

## Optional Phase 7 Runner

In [ ]:
RUN_PHASE07 = False

if RUN_PHASE07:
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices" / "run_15min_phase07_realistic_track_a.py"),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Phase 7 realistic Track A run failed with exit code {completed.returncode}.")

In [ ]:
latest_run = find_latest_phase07_run(config)
if latest_run is None:
    raise FileNotFoundError("No saved Phase 7 artifact exists yet. Run the phase 7 script first.")

status_summary = pd.read_csv(latest_run / "status_summary.csv")
bridge_summary = pd.read_csv(latest_run / "hourly_bridge_summary.csv")
feasibility = pd.read_csv(latest_run / "hourly_anchor_feasibility.csv")
external_coverage = pd.read_csv(latest_run / "external_feature_coverage.csv")
checks = pd.read_csv(latest_run / "validation_checks.csv")

hourly_metrics = pd.read_csv(latest_run / "hourly_anchor_metrics.csv") if (latest_run / "hourly_anchor_metrics.csv").exists() else pd.DataFrame()
hourly_coverage = pd.read_csv(latest_run / "hourly_anchor_coverage_summary.csv") if (latest_run / "hourly_anchor_coverage_summary.csv").exists() else pd.DataFrame()
price_metrics = pd.read_csv(latest_run / "reconstructed_price_metrics.csv") if (latest_run / "reconstructed_price_metrics.csv").exists() else pd.DataFrame()
shape_metrics = pd.read_csv(latest_run / "shape_only_metrics.csv") if (latest_run / "shape_only_metrics.csv").exists() else pd.DataFrame()
comparison = pd.read_csv(latest_run / "oracle_vs_realistic_comparison.csv") if (latest_run / "oracle_vs_realistic_comparison.csv").exists() else pd.DataFrame()
recommendations = pd.read_csv(latest_run / "recommended_model_summary.csv") if (latest_run / "recommended_model_summary.csv").exists() else pd.DataFrame()
run_summary = json.loads((latest_run / "run_summary.json").read_text(encoding="utf-8"))

display(pd.DataFrame([{"latest_phase07_run": str(latest_run)}]))

## Phase 7 Status

In [ ]:
display(status_summary)

## Hourly Bridge Summary

In [ ]:
display(bridge_summary)

## Hourly Anchor Feasibility

In [ ]:
display(feasibility)

## Blocking External Feature Coverage

In [ ]:
blocking = external_coverage[external_coverage["covers_requested_test_end"].fillna(False) == False].copy()
display(blocking.head(80))

## Validation Checks

In [ ]:
display(checks)

## Hourly Anchor Metrics

In [ ]:
display(hourly_metrics)

## Hourly Anchor Coverage

In [ ]:
display(hourly_coverage)

## Realistic 15-Minute Price Metrics

In [ ]:
display(price_metrics)

## Realistic Shape Metrics

In [ ]:
display(shape_metrics)

## Oracle vs Realistic Comparison

In [ ]:
display(comparison)

## Recommended Models

In [ ]:
display(recommendations)